In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import math
import time
import json
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, Subset

# 1. Setup Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

# 2. DualBN and AFA Definitions
class DualBN2d(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.bn_main = nn.BatchNorm2d(num_features)
        self.bn_aux = nn.BatchNorm2d(num_features)
        self.branch = 'main'
    def forward(self, x):
        return self.bn_main(x) if self.branch == 'main' else self.bn_aux(x)

def convert_to_dualbn(module):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, DualBN2d(child.num_features))
        else:
            convert_to_dualbn(child)
    return module

def set_branch(model, branch):
    for m in model.modules():
        if isinstance(m, DualBN2d):
            m.branch = branch

def fourier_basis(H, W, f, omega, device):
    u = torch.arange(H, device=device).view(H, 1).float()
    v = torch.arange(W, device=device).view(1, W).float()
    wave = torch.sin(2 * math.pi * f * (u * math.cos(omega) + v * math.sin(omega)) - math.pi / 4)
    return wave / wave.norm(p=2)

def afa_augment(images, inv_lambda=8.0):
    B, C, H, W = images.shape
    dev = images.device
    M = max(H, W)
    out = images.clone()
    exp_dist = torch.distributions.Exponential(1.0 / inv_lambda)
    for b in range(B):
        for c in range(C):
            f = torch.empty(1).uniform_(1, M).item()
            omega = torch.empty(1).uniform_(0, math.pi).item()
            sigma = exp_dist.sample().item()
            wave = fourier_basis(H, W, f, omega, dev)
            out[b, c] = images[b, c] + sigma * wave
    return torch.clamp(out, 0.0, 1.0)

# 3. Model Builder Helper (With CIFAR-10 stem fix)
def build_resnet18(use_dualbn=False):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    if use_dualbn:
        model = convert_to_dualbn(model)
    return model.to(device)

# 4. Data Preparation
DATA_DIR = "/kaggle/working/data"
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])
test_transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

# 5. Data-Size Scaling Loop
fractions = [0.10, 0.25, 0.50]
NUM_EPOCHS = 30
np.random.seed(42)
full_size = len(train_dataset)

scaling_results = {}

print("\n=== STARTING STAGE 4 DATA-SIZE SCALING EXPERIMENT ===")

for frac in fractions:
    subset_size = int(frac * full_size)
    indices = np.random.choice(full_size, size=subset_size, replace=False)
    sub_dataset = Subset(train_dataset, indices)
    train_loader = DataLoader(sub_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    print(f"\n---> Running Data Fraction: {int(frac*100)}% ({subset_size} images) <---")
    
    # A. Train Baseline Model
    baseline_model = build_resnet18(use_dualbn=False)
    opt_base = torch.optim.SGD(baseline_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    sched_base = torch.optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(NUM_EPOCHS):
        baseline_model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt_base.zero_grad()
            loss = criterion(baseline_model(imgs), lbls)
            loss.backward()
            opt_base.step()
        sched_base.step()
    
    # Evaluate Baseline Clean Test Accuracy
    baseline_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = baseline_model(imgs).argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
    clean_acc_base = 100.0 * correct / total
    
    # B. Train AFA Model
    afa_model = build_resnet18(use_dualbn=True)
    opt_afa = torch.optim.SGD(afa_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    sched_afa = torch.optim.lr_scheduler.CosineAnnealingLR(opt_afa, T_max=NUM_EPOCHS)
    
    for epoch in range(NUM_EPOCHS):
        afa_model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            afa_imgs = afa_augment(imgs, inv_lambda=8.0)
            
            opt_afa.zero_grad()
            set_branch(afa_model, 'main')
            loss_m = criterion(afa_model(imgs), lbls)
            
            set_branch(afa_model, 'aux')
            loss_a = criterion(afa_model(afa_imgs), lbls)
            
            loss_ace = 0.5 * (loss_m + loss_a)
            loss_ace.backward()
            opt_afa.step()
        sched_afa.step()
        
    # Evaluate AFA Clean Test Accuracy
    afa_model.eval()
    set_branch(afa_model, 'main')
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = afa_model(imgs).argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
    clean_acc_afa = 100.0 * correct / total
    
    scaling_results[f"{int(frac*100)}%"] = {
        "samples": subset_size,
        "baseline_clean_acc": round(clean_acc_base, 2),
        "afa_clean_acc": round(clean_acc_afa, 2)
    }
    print(f"Fraction {int(frac*100)}% Complete! -> Baseline: {clean_acc_base:.2f}% | AFA: {clean_acc_afa:.2f}%")

print("\n=== EXPERIMENT COMPLETE ===")
print(json.dumps(scaling_results, indent=2))

Using device: cuda


100%|██████████| 170M/170M [15:53<00:00, 179kB/s]



=== STARTING STAGE 4 DATA-SIZE SCALING EXPERIMENT ===

---> Running Data Fraction: 10% (5000 images) <---
Fraction 10% Complete! -> Baseline: 66.68% | AFA: 57.42%

---> Running Data Fraction: 25% (12500 images) <---
Fraction 25% Complete! -> Baseline: 84.05% | AFA: 83.32%

---> Running Data Fraction: 50% (25000 images) <---
Fraction 50% Complete! -> Baseline: 89.92% | AFA: 90.15%

=== EXPERIMENT COMPLETE ===
{
  "10%": {
    "samples": 5000,
    "baseline_clean_acc": 66.68,
    "afa_clean_acc": 57.42
  },
  "25%": {
    "samples": 12500,
    "baseline_clean_acc": 84.05,
    "afa_clean_acc": 83.32
  },
  "50%": {
    "samples": 25000,
    "baseline_clean_acc": 89.92,
    "afa_clean_acc": 90.15
  }
}


In [3]:
import os
import math
import time
import json
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, Subset

# 1. Device Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

# 2. DualBN and Vectorized CUDA Fourier Noise Generator
class DualBN2d(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.bn_main = nn.BatchNorm2d(num_features)
        self.bn_aux = nn.BatchNorm2d(num_features)
        self.branch = 'main'
    def forward(self, x):
        return self.bn_main(x) if self.branch == 'main' else self.bn_aux(x)

def convert_to_dualbn(module):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, DualBN2d(child.num_features))
        else:
            convert_to_dualbn(child)
    return module

def set_branch(model, branch):
    for m in model.modules():
        if isinstance(m, DualBN2d):
            m.branch = branch

def afa_augment_vectorized(images, inv_lambda=8.0):
    B, C, H, W = images.shape
    dev = images.device
    M = max(H, W)
    
    f = torch.empty(B, C, 1, 1, device=dev).uniform_(1, M)
    omega = torch.empty(B, C, 1, 1, device=dev).uniform_(0, math.pi)
    exp_dist = torch.distributions.Exponential(1.0 / inv_lambda)
    sigma = exp_dist.sample((B, C, 1, 1)).to(dev)
    
    u = torch.arange(H, device=dev).float().view(1, 1, H, 1)
    v = torch.arange(W, device=dev).float().view(1, 1, 1, W)
    
    arg = 2 * math.pi * f * (u * torch.cos(omega) + v * torch.sin(omega)) - (math.pi / 4)
    wave = torch.sin(arg)
    norms = torch.norm(wave, p=2, dim=(2, 3), keepdim=True)
    wave = wave / (norms + 1e-8)
    
    out = images + sigma * wave
    return torch.clamp(out, 0.0, 1.0)

# 3. Model Builder (CIFAR-10 Stem Fix)
def build_resnet18(use_dualbn=False):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    if use_dualbn:
        model = convert_to_dualbn(model)
    return model.to(device)

# 4. Data Setup
DATA_DIR = "/kaggle/working/data"
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])
test_transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

# 5. Data-Size Scaling Loop at 100 Epochs
fractions = [0.10, 0.25, 0.50, 1.00]  # All 4 percentages
NUM_EPOCHS = 100
np.random.seed(42)
full_size = len(train_dataset)

results_100epochs = {}

print("\n=== STARTING 100-EPOCH DATA-SIZE SCALING EXPERIMENT (10%, 25%, 50%, 100%) ===")

for frac in fractions:
    subset_size = int(frac * full_size)
    if frac < 1.00:
        indices = np.random.choice(full_size, size=subset_size, replace=False)
        sub_dataset = Subset(train_dataset, indices)
    else:
        sub_dataset = train_dataset
        
    train_loader = DataLoader(sub_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
    criterion = nn.CrossEntropyLoss()
    
    print(f"\n================ Running {int(frac*100)}% Data ({subset_size} images) at 100 Epochs ================")
    
    # A. Baseline
    t0 = time.time()
    baseline_model = build_resnet18(use_dualbn=False)
    opt_base = torch.optim.SGD(baseline_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    sched_base = torch.optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=NUM_EPOCHS)
    
    for epoch in range(1, NUM_EPOCHS + 1):
        baseline_model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt_base.zero_grad()
            loss = criterion(baseline_model(imgs), lbls)
            loss.backward()
            opt_base.step()
        sched_base.step()
        
    baseline_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = baseline_model(imgs).argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
    base_acc = 100.0 * correct / total
    print(f"  Baseline ({int(frac*100)}%) Clean Acc: {base_acc:.2f}% (Time: {(time.time()-t0)/60:.1f} mins)")
    
    # B. AFA DualBN
    t0 = time.time()
    afa_model = build_resnet18(use_dualbn=True)
    opt_afa = torch.optim.SGD(afa_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    sched_afa = torch.optim.lr_scheduler.CosineAnnealingLR(opt_afa, T_max=NUM_EPOCHS)
    
    for epoch in range(1, NUM_EPOCHS + 1):
        afa_model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            afa_imgs = afa_augment_vectorized(imgs, inv_lambda=8.0)
            
            opt_afa.zero_grad()
            set_branch(afa_model, 'main')
            loss_m = criterion(afa_model(imgs), lbls)
            
            set_branch(afa_model, 'aux')
            loss_a = criterion(afa_model(afa_imgs), lbls)
            
            loss_ace = 0.5 * (loss_m + loss_a)
            loss_ace.backward()
            opt_afa.step()
        sched_afa.step()
        
    afa_model.eval()
    set_branch(afa_model, 'main')
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = afa_model(imgs).argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
    afa_acc = 100.0 * correct / total
    print(f"  AFA DualBN ({int(frac*100)}%) Clean Acc: {afa_acc:.2f}% (Time: {(time.time()-t0)/60:.1f} mins)")
    
    results_100epochs[f"{int(frac*100)}%"] = {
        "samples": subset_size,
        "baseline_clean_acc": round(base_acc, 2),
        "afa_clean_acc": round(afa_acc, 2)
    }

print("\n=== COMPLETE 100-EPOCH SCALING RESULTS ===")
print(json.dumps(results_100epochs, indent=2))

Using device: cuda

=== STARTING 100-EPOCH DATA-SIZE SCALING EXPERIMENT (10%, 25%, 50%, 100%) ===

================ Running 10% Data (5000 images) at 100 Epochs ================
  Baseline (10%) Clean Acc: 73.03% (Time: 7.1 mins)
  AFA DualBN (10%) Clean Acc: 71.16% (Time: 13.9 mins)

================ Running 25% Data (12500 images) at 100 Epochs ================
  Baseline (25%) Clean Acc: 86.09% (Time: 17.2 mins)
  AFA DualBN (25%) Clean Acc: 86.27% (Time: 34.1 mins)

================ Running 50% Data (25000 images) at 100 Epochs ================
  Baseline (50%) Clean Acc: 91.97% (Time: 34.3 mins)
  AFA DualBN (50%) Clean Acc: 91.93% (Time: 68.2 mins)

================ Running 100% Data (50000 images) at 100 Epochs ================
  Baseline (100%) Clean Acc: 95.12% (Time: 68.3 mins)
  AFA DualBN (100%) Clean Acc: 94.46% (Time: 136.0 mins)

=== COMPLETE 100-EPOCH SCALING RESULTS ===
{
  "10%": {
    "samples": 5000,
    "baseline_clean_acc": 73.03,
    "afa_clean_acc": 71.16
  },
 